# 01 — Explore ECOSoundSet
Inspect the dataset, understand class distribution, and identify which UK Orthoptera species have sufficient clips.

**Kernel:** `Python (orthoptera-training)`  
**Dataset:** `datasets/ecosoundset/` — download via `zenodo_get 15043892` if not present.

In [ ]:
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import os

DATASET_ROOT = "../../datasets/ecosoundset"
TRAIN_CSV    = os.path.join(DATASET_ROOT, "train_strong.csv")
AUDIO_DIR    = os.path.join(DATASET_ROOT, "audio")

In [ ]:
# ── Load the strong-label train split ────────────────────────────────────────
train = pd.read_csv(TRAIN_CSV)
print(f"Total clips: {len(train)}")
print(f"Columns:     {list(train.columns)}")
train.head()

In [ ]:
# ── All species in the dataset ────────────────────────────────────────────────
print(f"Species count: {train['species'].nunique()}\n")
print(train['species'].value_counts().to_string())

In [ ]:
# ── Filter to UK target species ───────────────────────────────────────────────
UK_SPECIES = [
    'Chorthippus brunneus',        # Field Grasshopper
    'Chorthippus parallelus',      # Meadow Grasshopper
    'Omocestus viridulus',         # Common Green Grasshopper
    'Tettigonia viridissima',      # Great Green Bush-cricket
    'Roeseliana roeselii',         # Roesel's Bush-cricket
    'Pholidoptera griseoaptera',   # Dark Bush-cricket
    'Leptophyes punctatissima',    # Speckled Bush-cricket
    'Meconema thalassinum',        # Oak Bush-cricket
    'Gryllus campestris',          # Field Cricket
]

uk_train = train[train['species'].isin(UK_SPECIES)].copy()
print(f"UK species clips: {len(uk_train)}\n")
print(uk_train['species'].value_counts().to_string())

In [ ]:
# ── Visualise class balance ───────────────────────────────────────────────────
counts = uk_train['species'].value_counts()
short_names = [s.split()[-1] for s in counts.index]  # genus only for readability

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(short_names, counts.values, color='#2d8a4e')
ax.set_xlabel("Clips")
ax.set_title("UK Orthoptera — clips per species (ECOSoundSet train split)")
ax.axvline(100, color='orange', linestyle='--', linewidth=1, label='100 clip threshold')
ax.legend()
plt.tight_layout()
plt.show()

# Flag species that may need augmentation or InsectSet459 top-up
low = counts[counts < 50]
if len(low):
    print(f"\n⚠ Species with <50 clips (consider InsectSet459 supplement):")
    print(low.to_string())

In [ ]:
# ── Inspect a clip — spectrogram preview ─────────────────────────────────────
sample = uk_train.iloc[0]
audio_path = os.path.join(AUDIO_DIR, sample['filename'])

if os.path.exists(audio_path):
    y, sr = librosa.load(audio_path, sr=None)
    print(f"Species:     {sample['species']}")
    print(f"File:        {sample['filename']}")
    print(f"Sample rate: {sr} Hz   Duration: {len(y)/sr:.1f} s")

    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"{sample['species']} — waveform")

    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=20000)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='mel',
                             fmax=20000, ax=axes[1], cmap='viridis')
    axes[1].set_title("Mel spectrogram")
    plt.colorbar(axes[1].collections[0], ax=axes[1], format='%+2.0f dB')
    plt.tight_layout()
    plt.show()
else:
    print(f"Audio file not found: {audio_path}")
    print("Download the dataset first — see setup guide Step 6.")